# CADEC G4 refit (LanguageTool)

CADEC’s original G4 run used the heuristic fallback (`g4_pass ≈ 0.996`).
This notebook re-scores **only G4** with real LanguageTool (`en-US`), using the
same `grammar_score` / `g4_pass` definition as
`RQ1_semantic_entropy_linguistic_predictors.ipynb`, then recomputes
`accepted_final` and rewrites both CADEC perturbation CSVs.

Does **not** regenerate perturbations or re-run G1–G3/G5/G6.

In [ ]:
# === Setup ===
import csv
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.home() / "projects" / "Measuring-Semantic-Stability-in-Clinical-LLMs"
INTER_DIR = PROJECT_ROOT / "outputs" / "rq3" / "intermediate"
FULL_CSV = INTER_DIR / "rq3_cadec_validated_perturbations_full.csv"
BIOASQ_CSV = INTER_DIR / "rq3_cadec_perturbations.csv"
BIOASQ_REF = INTER_DIR / "rq3_bioasq_perturbations.csv"

# LanguageTool needs a JRE. Prefer standalone Temurin (do NOT conda-install
# openjdk into torch_gpu — that can pull GraalPy and break CPython).
_java_candidates = [
    Path.home() / "data" / "jdk" / "temurin-17" / "bin" / "java",
    Path("/usr/bin/java"),
]
_java = next((p for p in _java_candidates if p.is_file()), None)
if _java is not None:
    _java_home = _java.parent.parent if _java.name == "java" else _java
    os.environ["JAVA_HOME"] = str(_java_home)
    os.environ["PATH"] = str(_java.parent) + os.pathsep + os.environ.get("PATH", "")
    print(f"Using Java: {_java}")
else:
    print("[WARN] No local java found — LanguageTool may fall back to remote API")


def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


for label, p in [("full validated", FULL_CSV), ("BioASQ schema ref", BIOASQ_REF)]:
    print(f"Path [{label}]: {p}")
    assert p.is_file(), f"Missing {label}: {p}"

TARGET_COLS = list(pd.read_csv(BIOASQ_REF, nrows=0).columns)
print("BioASQ pert schema:", TARGET_COLS)
_log("Setup OK.")

Using Java: /home/s224858267/data/jdk/temurin-17/bin/java
Path [full validated]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_validated_perturbations_full.csv
Path [BioASQ schema ref]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_perturbations.csv
BioASQ pert schema: ['instance_id', 'perturbation_type', 'mention_context', 'perturbation_text', 'lexical_change_magnitude', 'gold_mention', 'gold_cui', 'accepted_final']
[2026-07-29 00:58:23 UTC] Setup OK.


## 1) Load LanguageTool (must succeed — no heuristic fallback)

In [2]:
# === LanguageTool en-US (same as RQ1 intent) ===
import language_tool_python

try:
    _lt_tool = language_tool_python.LanguageTool("en-US")
    _probe = _lt_tool.check("This are wrong.")
    assert len(_probe) >= 1, "LanguageTool returned no matches on a known-bad sentence"
    print(f"✓ LanguageTool loaded (local). Probe matches={len(_probe)}")
except Exception as e_local:
    print(f"[WARN] Local LanguageTool failed: {e_local}")
    print("Trying public LanguageTool API (slower; rate-limited) …")
    _lt_tool = language_tool_python.LanguageTool(
        "en-US", remote_server="https://api.languagetool.org"
    )
    _probe = _lt_tool.check("This are wrong.")
    assert len(_probe) >= 1
    print(f"✓ LanguageTool loaded (remote API). Probe matches={len(_probe)}")

assert _lt_tool is not None
_log("LanguageTool ready — G4 will NOT use heuristic fallback.")

✓ LanguageTool loaded (local). Probe matches=2
[2026-07-29 00:58:31 UTC] LanguageTool ready — G4 will NOT use heuristic fallback.


## 2) RQ1-identical `grammar_score` + G4 pass rule

In [3]:
# === Copied from RQ1 six-gate cell (LanguageTool branch only) ==============
def grammar_score(text: str) -> float:
    """
    G4 grammaticality score using LanguageTool (pre-registered).
    Returns a score in [0, 1] where 1 = no grammar errors.

    Pre-registered threshold: score >= 0.5 to pass G4.
    (Equivalent to fewer than 1 grammar error per 10 tokens on average.)
    """
    # Refit notebook: LanguageTool MUST be available — do not silently heuristic.
    matches = _lt_tool.check(str(text))
    n_errors = len(matches)
    n_tokens = max(len(str(text).split()), 1)
    score = max(0.0, 1.0 - (n_errors / n_tokens))
    return float(score)


def g4_fields(mention_context: str, pert_text: str):
    """Match RQ1 validate_one G4 block exactly."""
    pert_text = str(pert_text)
    mention_context = str(mention_context)
    g4 = grammar_score(pert_text)
    g4_orig = grammar_score(mention_context)
    g4_delta = g4_orig - g4

    severe_malformed = (
        (len(pert_text.split()) < 2)
        or (sum(ch.isalpha() for ch in pert_text) / max(len(pert_text), 1) < 0.20)
        or (sum(ch in ".,:;!?()[]{}-" for ch in pert_text) / max(len(pert_text), 1) > 0.55)
    )
    # RQ1: g4_pass = (not severe_malformed) and (g4 >= 0.50)
    g4_pass = (not severe_malformed) and (g4 >= 0.50)
    return {
        "g4_grammar_score": g4,
        "g4_original_score": g4_orig,
        "g4_score_delta": g4_delta,
        "g4_pass": bool(g4_pass),
    }


# Quick unit check on known-good / known-bad
_bad = grammar_score("This are wrong.")
_good = grammar_score("This is correct.")
print(f"Probe scores: bad={_bad:.3f} good={_good:.3f}")
assert _bad < _good, "LanguageTool scores do not discriminate bad vs good text"
_log("G4 helpers ready.")

Probe scores: bad=0.333 good=1.000
[2026-07-29 00:58:31 UTC] G4 helpers ready.


## 3) Load CSV, snapshot before stats, refit G4

In [4]:
# === Load + before snapshot =================================================
print(f"Reading: {FULL_CSV}")
df = pd.read_csv(FULL_CSV, low_memory=False)
_log(f"Loaded {len(df):,} rows")
assert len(df) == 55_976, f"Expected 55,976 rows, got {len(df):,}"

GATE_COLS = ["g1_pass", "g2_pass", "g3_pass", "g4_pass", "g5_pass", "g6_pass"]
for c in GATE_COLS + ["accepted_final", "perturbation_text", "mention_context"]:
    assert c in df.columns, f"Missing column {c}"


def _bool_rate(s):
    return float(pd.Series(s).fillna(False).astype(bool).mean())


def snapshot(frame, label):
    print(f"\n===== {label} =====")
    print(f"overall acceptance: {_bool_rate(frame['accepted_final']):.4f} "
          f"({int(frame['accepted_final'].fillna(False).astype(bool).sum()):,} / {len(frame):,})")
    print("per-gate pass rates:")
    for g in GATE_COLS:
        print(f"  {g:10s} {_bool_rate(frame[g]):.4f}")
    print("per-type acceptance:")
    rates = (
        frame.groupby("perturbation_type", as_index=False)
        .agg(n=("accepted_final", "size"), n_accepted=("accepted_final", lambda x: int(pd.Series(x).fillna(False).astype(bool).sum())))
    )
    rates["acceptance_rate"] = rates["n_accepted"] / rates["n"].clip(lower=1)
    print(rates.sort_values("perturbation_type").to_string(index=False))
    sys.stdout.flush()
    return {
        "acceptance": _bool_rate(frame["accepted_final"]),
        "g4_pass": _bool_rate(frame["g4_pass"]),
        "g4_score_mean": float(frame["g4_grammar_score"].astype(float).mean()),
    }


# Keep pre-refit G4 for flip diagnostics (dropped before save)
df["_g4_pass_before"] = df["g4_pass"]
df["_g4_score_before"] = df["g4_grammar_score"]

before = snapshot(df, "BEFORE (heuristic G4)")
assert before["g4_pass"] > 0.99, (
    f"Expected heuristic-era g4_pass ~0.996, got {before['g4_pass']:.4f}"
)

Reading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_validated_perturbations_full.csv


[2026-07-29 00:58:32 UTC] Loaded 55,976 rows



===== BEFORE (heuristic G4) =====
overall acceptance: 0.5389 (30,168 / 55,976)
per-gate pass rates:
  g1_pass    0.9207
  g2_pass    0.9874
  g3_pass    0.9765
  g4_pass    0.9958
  g5_pass    0.6078
  g6_pass    0.9325
per-type acceptance:
    perturbation_type     n  n_accepted  acceptance_rate
     back_translation 13994        9128         0.652280
controlled_paraphrase 13994        7618         0.544376
 synonym_substitution 13994       12064         0.862084
 syntactic_reordering 13994        1358         0.097042


In [5]:
# === Refit G4 over unique texts (cache) then map back =======================
from tqdm.auto import tqdm

# Cache grammar_score by exact string — mention_context + perturbation_text
unique_texts = pd.unique(
    pd.concat(
        [df["perturbation_text"].astype(str), df["mention_context"].astype(str)],
        ignore_index=True,
    )
)
_log(f"Unique texts to score with LanguageTool: {len(unique_texts):,}")

score_cache = {}
for t in tqdm(unique_texts, desc="LanguageTool G4"):
    score_cache[t] = grammar_score(t)

_log(f"Cached {len(score_cache):,} grammar scores")

g4_scores = []
g4_origs = []
g4_deltas = []
g4_passes = []

for mc, pt in zip(df["mention_context"].astype(str), df["perturbation_text"].astype(str)):
    g4 = score_cache[pt]
    g4_orig = score_cache[mc]
    g4_delta = g4_orig - g4
    severe_malformed = (
        (len(pt.split()) < 2)
        or (sum(ch.isalpha() for ch in pt) / max(len(pt), 1) < 0.20)
        or (sum(ch in ".,:;!?()[]{}-" for ch in pt) / max(len(pt), 1) > 0.55)
    )
    g4_pass = (not severe_malformed) and (g4 >= 0.50)
    g4_scores.append(g4)
    g4_origs.append(g4_orig)
    g4_deltas.append(g4_delta)
    g4_passes.append(bool(g4_pass))

df["g4_grammar_score"] = g4_scores
df["g4_original_score"] = g4_origs
df["g4_score_delta"] = g4_deltas
df["g4_pass"] = g4_passes

_log(
    f"G4 refit done | g4_pass={_bool_rate(df['g4_pass']):.4f} "
    f"| mean score={df['g4_grammar_score'].mean():.4f}"
)

[2026-07-29 00:58:32 UTC] Unique texts to score with LanguageTool: 26,454


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



LanguageTool G4:   0%|          | 0/26454 [00:00<?, ?it/s]


LanguageTool G4:   0%|          | 2/26454 [00:00<26:08, 16.86it/s]


LanguageTool G4:   0%|          | 4/26454 [00:00<25:44, 17.13it/s]

...


LanguageTool G4: 100%|█████████▉| 26451/26454 [19:53<00:00, 26.77it/s]


LanguageTool G4: 100%|██████████| 26454/26454 [19:53<00:00, 20.37it/s]


LanguageTool G4: 100%|██████████| 26454/26454 [19:53<00:00, 22.16it/s]

[2026-07-29 01:18:26 UTC] Cached 26,454 grammar scores


[2026-07-29 01:18:27 UTC] G4 refit done | g4_pass=0.9871 | mean score=0.8947


## 4) Recompute `accepted_final` (RQ1 conjunction) + after stats

In [6]:
# === accepted_final — RQ1 validate_one conjunction ==========================
# RQ1: accepted = all([g1_pass, g2_pass, g3_pass, g4_pass, g5_pass, g6_pass, noop_pass])
# noop_pass encoded as fallback_noop_rejected == 0 when that column exists.

def _as_bool(s):
    return pd.Series(s).fillna(False).astype(bool)


conj = (
    _as_bool(df["g1_pass"])
    & _as_bool(df["g2_pass"])
    & _as_bool(df["g3_pass"])
    & _as_bool(df["g4_pass"])
    & _as_bool(df["g5_pass"])
    & _as_bool(df["g6_pass"])
)
if "fallback_noop_rejected" in df.columns:
    noop_pass = df["fallback_noop_rejected"].fillna(0).astype(int).eq(0)
    df["accepted_final"] = conj & noop_pass
    _log("accepted_final = G1∧G2∧G3∧G4∧G5∧G6 ∧ ¬fallback_noop_rejected (RQ1 exact)")
else:
    df["accepted_final"] = conj
    _log("accepted_final = G1∧G2∧G3∧G4∧G5∧G6")

# Diversity-rescue flags from the original run are stale once G4 changes
if "accepted_relaxed" in df.columns:
    df["accepted_relaxed"] = False

after = snapshot(df, "AFTER (LanguageTool G4)")

print("\n===== DELTA =====")
print(f"acceptance: {before['acceptance']:.4f} → {after['acceptance']:.4f} "
      f"(Δ={after['acceptance'] - before['acceptance']:+.4f})")
print(f"g4_pass:    {before['g4_pass']:.4f} → {after['g4_pass']:.4f} "
      f"(Δ={after['g4_pass'] - before['g4_pass']:+.4f})")
print(f"g4_score μ: {before['g4_score_mean']:.4f} → {after['g4_score_mean']:.4f}")
sys.stdout.flush()

# Assert LanguageTool actually changed G4. Heuristic scores clustered ~0.92 with
# pass≈0.996; real LT can still have a high pass rate (MedMentions ~0.9975), so
# require score-distribution movement and/or pass flips — not pass<0.99.
_n_flip = int((_as_bool(df["g4_pass"]) != _as_bool(df["_g4_pass_before"])).sum())
_score_delta = abs(after["g4_score_mean"] - before["g4_score_mean"])
_pass_delta = abs(after["g4_pass"] - before["g4_pass"])
print(f"G4 pass flips: {_n_flip:,} | score mean Δ={_score_delta:.4f} | pass Δ={_pass_delta:.4f}")
assert _n_flip > 0 or _score_delta > 1e-3 or _pass_delta > 1e-4, (
    "G4 unchanged after LanguageTool refit — still identical to heuristic run."
)
df.drop(columns=["_g4_pass_before", "_g4_score_before"], inplace=True, errors="ignore")
_log("ASSERT OK: G4 refit applied LanguageTool scores (not unchanged heuristic).")

[2026-07-29 01:18:27 UTC] accepted_final = G1∧G2∧G3∧G4∧G5∧G6 ∧ ¬fallback_noop_rejected (RQ1 exact)



===== AFTER (LanguageTool G4) =====
overall acceptance: 0.5336 (29,868 / 55,976)
per-gate pass rates:
  g1_pass    0.9207
  g2_pass    0.9874
  g3_pass    0.9765
  g4_pass    0.9871
  g5_pass    0.6078
  g6_pass    0.9325
per-type acceptance:
    perturbation_type     n  n_accepted  acceptance_rate
     back_translation 13994        9132         0.652565
controlled_paraphrase 13994        7554         0.539803
 synonym_substitution 13994       11842         0.846220
 syntactic_reordering 13994        1340         0.095755



===== DELTA =====
acceptance: 0.5389 → 0.5336 (Δ=-0.0054)
g4_pass:    0.9958 → 0.9871 (Δ=-0.0087)
g4_score μ: 0.9187 → 0.8947


G4 pass flips: 524 | score mean Δ=0.0240 | pass Δ=0.0087
[2026-07-29 01:18:27 UTC] ASSERT OK: G4 refit applied LanguageTool scores (not unchanged heuristic).


## 5) Save full CSV + BioASQ-schema export

In [7]:
# === Write outputs ==========================================================
print(f"Writing full: {FULL_CSV}")
df.to_csv(
    FULL_CSV,
    index=False,
    quoting=csv.QUOTE_MINIMAL,
    escapechar="\\",
)
_log(f"Wrote {len(df):,} rows → {FULL_CSV}")

missing = [c for c in TARGET_COLS if c not in df.columns]
assert not missing, f"Missing BioASQ columns: {missing}"
df_export = df[TARGET_COLS].copy()
print(f"Writing BioASQ schema: {BIOASQ_CSV}")
df_export.to_csv(
    BIOASQ_CSV,
    index=False,
    quoting=csv.QUOTE_MINIMAL,
    escapechar="\\",
)
_log(f"Wrote {len(df_export):,} rows → {BIOASQ_CSV}")
print("Export columns:", list(df_export.columns))
assert list(df_export.columns) == TARGET_COLS
assert int(df_export["accepted_final"].fillna(False).astype(bool).sum()) > 0
_log("Done.")

# Close LanguageTool server if local
try:
    _lt_tool.close()
except Exception:
    pass

Writing full: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_validated_perturbations_full.csv


[2026-07-29 01:18:29 UTC] Wrote 55,976 rows → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_validated_perturbations_full.csv


Writing BioASQ schema: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_perturbations.csv


[2026-07-29 01:18:30 UTC] Wrote 55,976 rows → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_perturbations.csv


Export columns: ['instance_id', 'perturbation_type', 'mention_context', 'perturbation_text', 'lexical_change_magnitude', 'gold_mention', 'gold_cui', 'accepted_final']
[2026-07-29 01:18:30 UTC] Done.
